In [ ]:
"""
Phase 3: Data Cleaning and Preprocessing — Wired vs The Verge
=============================================================
Inputs : data/raw/wired_posts_raw.csv, data/raw/theverge_posts_raw.csv
         data/raw/wired_profiles_raw.csv, data/raw/theverge_profiles_raw.csv
Outputs: data/processed/wired_posts_clean.csv, data/processed/theverge_posts_clean.csv
         data/processed/wired_profiles_clean.csv, data/processed/theverge_profiles_clean.csv
         data/processed/cleaning_summary.json

The cleaning pipeline mirrors Week 7's approach with three additions:
  1. Extract URLs / domains / mentions / hashtags BEFORE stripping them.
  2. Maintain three text columns — raw, cleaned, and cleaned-without-brand-tokens.
  3. Brand-relevance check to remove residual idiom false positives that
     slipped through the precision-first query design (notably "on the verge of"
     which the phrase query "the verge" can match in non-brand contexts, and
     adjective uses of "wired" e.g. "fully wired", "wired in").
"""
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from textblob import Word

for pkg in ["stopwords", "wordnet", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(
            f"corpora/{pkg}" if pkg != "punkt" and pkg != "punkt_tab"
            else f"tokenizers/{pkg}"
        )
    except LookupError:
        nltk.download(pkg, quiet=True)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR  = PROJECT_ROOT / "data" / "raw"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading raw data from   : {RAW_DIR}")
print(f"Writing processed data to: {PROC_DIR}")

Reading raw data from   : /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/raw
Writing processed data to: /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/processed


In [2]:
wired_posts_raw = pd.read_csv(RAW_DIR / "wired_posts_raw.csv")
verge_posts_raw = pd.read_csv(RAW_DIR / "theverge_posts_raw.csv")

wired_profiles_raw = pd.read_csv(RAW_DIR / "wired_profiles_raw.csv")
verge_profiles_raw = pd.read_csv(RAW_DIR / "theverge_profiles_raw.csv")

print(f"Wired posts (raw)        : {len(wired_posts_raw):>5}")
print(f"The Verge posts (raw)    : {len(verge_posts_raw):>5}")
print(f"Wired profiles (raw)     : {len(wired_profiles_raw):>5}")
print(f"The Verge profiles (raw) : {len(verge_profiles_raw):>5}")

Wired posts (raw)        :  1776
The Verge posts (raw)    :  1946
Wired profiles (raw)     :  1465
The Verge profiles (raw) :  1514


In [3]:
URL_RE     = re.compile(r"https?://\S+|www\.\S+|\S+\.(?:com|co|io|net|org|ai|app|dev|gg)/\S*", re.IGNORECASE)
MENTION_RE = re.compile(r"@[\w.-]+")
HASHTAG_RE = re.compile(r"#\w+")
DOMAIN_RE  = re.compile(r"(?:https?://)?(?:www\.)?([\w-]+(?:\.[\w-]+)+)", re.IGNORECASE)

def extract_features(text: str) -> dict:
    if not isinstance(text, str):
        return {"urls": [], "mentions": [], "hashtags": [], "domains": []}
    urls     = URL_RE.findall(text)
    mentions = MENTION_RE.findall(text)
    hashtags = [h.lower() for h in HASHTAG_RE.findall(text)]
    domains  = [m.group(1).lower() for m in DOMAIN_RE.finditer(text) if m.group(1)]
    domains  = [d for d in domains if d.count(".") >= 1 and len(d) >= 4]
    return {"urls": urls, "mentions": mentions, "hashtags": hashtags, "domains": domains}

def add_extracted_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = df["text"].apply(extract_features).apply(pd.Series)
    df = df.copy()
    df["urls"]       = feats["urls"].apply(lambda xs: ";".join(xs))
    df["mentions"]   = feats["mentions"].apply(lambda xs: ";".join(xs))
    df["hashtags"]   = feats["hashtags"].apply(lambda xs: ";".join(xs))
    df["domains"]    = feats["domains"].apply(lambda xs: ";".join(xs))
    df["n_urls"]     = feats["urls"].apply(len)
    df["n_mentions"] = feats["mentions"].apply(len)
    df["n_hashtags"] = feats["hashtags"].apply(len)
    return df

wired_posts = add_extracted_features(wired_posts_raw)
verge_posts = add_extracted_features(verge_posts_raw)
print("Extracted URL/mention/hashtag/domain features.")

Extracted URL/mention/hashtag/domain features.


In [4]:
# Brand-relevance filter — applied at the cleaning stage rather than collection,
# because the precision-first phrase queries already enforced strong relevance
# at search time. The remaining concern is the English idiom "on the verge of"
# (also "verge of collapse", "verge of tears") which the phrase query "the verge"
# admits as false positives. We also do a defensive pass on Wired to catch any
# "fully wired" / "wired in" / "hard-wired" adjective uses.
#
# Logic for each brand:
#   Path 1: explicit domain reference        -> always relevant
#   Path 2: explicit handle reference        -> always relevant
#   Path 3: brand name + journalism anchor   -> relevant
#   Otherwise: reject as adjective/idiom usage

# Domain / handle anchors
WIRED_DOMAIN = re.compile(r"wired\.com|@wired\.com|wired\.co\.uk", flags=re.IGNORECASE)
VERGE_DOMAIN = re.compile(r"theverge\.com|@theverge\.com",          flags=re.IGNORECASE)

# Journalism / publication-context anchors. Co-occurrence with the brand name
# strongly implies the publication-as-brand reading vs the adjective/idiom.
PUB_ANCHORS = re.compile(
    r"\b(article|piece|story|stories|published|publish|publishes|"
    r"reports?|reported|reporting|writes?|wrote|writing|writer|"
    r"journalist|reviews?|reviewed|reviewer|interview|interviewed|"
    r"covers?|covered|coverage|feature|featured|column|columnist|"
    r"editorial|editor|magazine|publication|byline|headline|"
    r"investigation|investigates?|investigated|"
    r"podcast|newsletter|subscribe|subscription|paywall|"
    r"according\s+to|via)\b",
    flags=re.IGNORECASE,
)

# Capitalised proper-noun patterns. Case-sensitive — "wired" lowercase is the
# adjective; "Wired" capitalised is the brand. "verge" lowercase + "the" can be
# either; capitalised "Verge" is more reliable.
WIRED_PROPER     = re.compile(r"\bWired\b")
VERGE_PROPER     = re.compile(r"\bVerge\b|\bThe Verge\b")
THE_VERGE_LOWER  = re.compile(r"\bthe verge\b", flags=re.IGNORECASE)

def is_brand_relevant_wired(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    # Path 1+2: domain or handle reference
    if WIRED_DOMAIN.search(text):
        return True
    # Path 3: capitalised "Wired" + a journalism anchor
    if WIRED_PROPER.search(text) and PUB_ANCHORS.search(text):
        return True
    return False

def is_brand_relevant_verge(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    # Path 1+2: domain or handle reference (the strongest signal)
    if VERGE_DOMAIN.search(text):
        return True
    # Path 3: capitalised "Verge" + journalism anchor
    if VERGE_PROPER.search(text) and PUB_ANCHORS.search(text):
        return True
    # Path 4: lowercase "the verge" + journalism anchor (catches casual posts)
    if THE_VERGE_LOWER.search(text) and PUB_ANCHORS.search(text):
        return True
    return False

# Apply
wired_before = len(wired_posts)
wired_posts  = wired_posts[wired_posts["text"].apply(is_brand_relevant_wired)].reset_index(drop=True)
print(f"Wired brand-relevance     : {wired_before} -> {len(wired_posts)} posts retained")

verge_before = len(verge_posts)
verge_posts  = verge_posts[verge_posts["text"].apply(is_brand_relevant_verge)].reset_index(drop=True)
print(f"The Verge brand-relevance : {verge_before} -> {len(verge_posts)} posts retained")

Wired brand-relevance     : 1776 -> 1187 posts retained
The Verge brand-relevance : 1946 -> 1250 posts retained


In [5]:
# Stopword set: NLTK English + custom additions for Bluesky/journalism discourse.
# 'wired', 'verge' etc. are added to a *separate* set so we can remove them only
# for topic modelling (where they would dominate as the topic), not for general
# keyword frequency or sentiment.

BASE_STOPWORDS = set(stopwords.words("english"))

CUSTOM_STOPWORDS = {
    # generic web/Bluesky noise
    "rt", "amp", "via", "http", "https", "com", "www", "html",
    "bsky", "app", "post", "thread", "skeet",
    # filler
    "like", "get", "got", "go", "going", "use", "using", "used",
    "one", "two", "would", "could", "also", "really", "much", "many",
    "see", "say", "said", "make", "made", "take", "took",
    "want", "wanted", "even", "still", "well", "way", "things", "thing",
    "today", "yesterday", "tomorrow", "now", "ll", "ve", "re",
}

# Brand tokens — only stripped for topic-modelling text variant
BRAND_STOPWORDS = {
    "wired", "wiredcom", "wiredmag",
    "verge", "theverge", "thevergecom",
}

ALL_STOPWORDS          = BASE_STOPWORDS | CUSTOM_STOPWORDS
ALL_STOPWORDS_NO_BRAND = ALL_STOPWORDS  | BRAND_STOPWORDS

def clean_text(text: str) -> str:
    """
    Course-aligned aggressive clean (mirrors Week 7's cleanText):
      - strip URLs / mentions / hashtags (we already extracted them)
      - lowercase
      - remove punctuation, digits, single letters
      - normalise accented characters
    """
    if not isinstance(text, str):
        return ""
    # Bluesky-specific strips first
    text = URL_RE.sub(" ",     text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    # Course-style cleaning
    text = re.sub(r"[^\w\s]", " ", text)       # remove punctuation
    text = re.sub(r"\d+", " ", text)           # remove digits
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)  # remove single letters
    text = text.lower()
    text = re.sub(u"[àáâãäå]", "a", text)
    text = re.sub(u"[èéêë]",   "e", text)
    text = re.sub(u"[ìíîï]",   "i", text)
    text = re.sub(u"[òóôõö]",  "o", text)
    text = re.sub(u"[ùúûü]",   "u", text)
    text = re.sub(u"[ț]",      "t", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stopwords_lemmatize(text: str, stopword_set: set) -> str:
    """Token-level pass: drop stopwords and lemmatize what survives."""
    if not text:
        return ""
    tokens = [
        Word(tok).lemmatize()
        for tok in text.split()
        if tok not in stopword_set and len(tok) > 2
    ]
    return " ".join(tokens)

def light_clean_for_sentiment(text: str) -> str:
    """
    Lighter clean for VADER / TextBlob sentiment — keeps casing, punctuation,
    emoji-style cues, but strips URLs/mentions/hashtags as those don't carry
    affect and inflate the input length.
    """
    if not isinstance(text, str):
        return ""
    text = URL_RE.sub(" ",     text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
def clean_posts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.rename(columns={"text": "text_raw"})

    # Light clean — for sentiment scoring (VADER prefers natural text)
    df["text_for_sentiment"] = df["text_raw"].apply(light_clean_for_sentiment)

    # Aggressive clean — for keyword/frequency analysis
    df["text_clean_stage1"] = df["text_raw"].apply(clean_text)
    df["text_clean"] = df["text_clean_stage1"].apply(
        lambda t: remove_stopwords_lemmatize(t, ALL_STOPWORDS)
    )

    # Topic-modelling variant — same as text_clean but brand tokens removed
    df["text_clean_no_brand"] = df["text_clean_stage1"].apply(
        lambda t: remove_stopwords_lemmatize(t, ALL_STOPWORDS_NO_BRAND)
    )

    # Word counts (used in analysis: post-length distribution, engagement vs length)
    df["word_count_raw"]   = df["text_raw"].fillna("").str.split().str.len()
    df["word_count_clean"] = df["text_clean"].str.split().str.len()

    # Drop the intermediate stage column to keep CSV tidy
    df = df.drop(columns=["text_clean_stage1"])
    return df

wired_posts = clean_posts(wired_posts)
verge_posts = clean_posts(verge_posts)
print("Applied three-tier text cleaning to both brands.")
wired_posts[["text_raw", "text_clean", "text_clean_no_brand"]].head(3)

Applied three-tier text cleaning to both brands.


,text_raw,text_clean,text_clean_no_brand
0,The #CIA #Covert Operation: Episode 2: The Ext...,operation episode extraction citizen,operation episode extraction citizen
1,#EpsteinClass \n@eff.org \n@wired.com \n@indiv...,,
2,@wired.com WHAT DO YOU MEAN THIS IS MY LAST FR...,mean last free article site month,mean last free article site month


In [7]:
def filter_posts(df: pd.DataFrame, brand_label: str) -> tuple[pd.DataFrame, dict]:
    """
    Apply quality filters and record what was removed at each step.
    Returns (filtered_df, audit_dict) for transparent reporting.
    """
    audit = {"start": len(df)}

    # 1. Language — keep English-only
    if "langs" in df.columns:
        df = df[df["langs"].fillna("").str.contains("en", case=False, regex=False)]
    audit["after_language"] = len(df)

    # 2. Deduplicate by URI (defensive — already done in collection)
    df = df.drop_duplicates(subset="uri", keep="first")
    audit["after_uri_dedup"] = len(df)

    # 3. Deduplicate near-identical text from the same author (likely repost-spam)
    df = df.drop_duplicates(subset=["author_did", "text_raw"], keep="first")
    audit["after_author_text_dedup"] = len(df)

    # 4. Length filter — remove link-only or near-empty posts
    df = df[df["word_count_clean"] >= 3]
    audit["after_min_length"] = len(df)

    # 5. Spam signal: same author posting >5 times with identical cleaned text
    spam_mask = df.groupby(["author_did", "text_clean"])["uri"].transform("count") > 5
    df = df[~spam_mask]
    audit["after_spam_filter"] = len(df)

    # 6. Parse timestamps now (will be needed in analysis)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    df = df.dropna(subset=["created_at"])
    audit["after_timestamp_parse"] = len(df)

    df = df.reset_index(drop=True)
    audit["final"] = len(df)

    print(f"\n[{brand_label}] filtering audit:")
    for k, v in audit.items():
        print(f"  {k:<28} {v:>5}")
    return df, audit

wired_posts, wired_audit = filter_posts(wired_posts, "Wired")
verge_posts, verge_audit = filter_posts(verge_posts, "The Verge")


[Wired] filtering audit:
  start                         1187
  after_language                1187
  after_uri_dedup               1187
  after_author_text_dedup       1167
  after_min_length              1136
  after_spam_filter             1136
  after_timestamp_parse         1090
  final                         1090

[The Verge] filtering audit:
  start                         1250
  after_language                1250
  after_uri_dedup               1250
  after_author_text_dedup       1246
  after_min_length              1167
  after_spam_filter             1167
  after_timestamp_parse         1156
  final                         1156


In [8]:
def clean_profiles(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop_duplicates(subset="did", keep="first")

    # Coerce numeric columns and parse account creation date
    for col in ["followers_count", "follows_count", "posts_count"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)

    # Engagement-relevant ratio — pre-computed for influencer scoring later
    df["follower_following_ratio"] = np.where(
        df["follows_count"] > 0,
        df["followers_count"] / df["follows_count"],
        df["followers_count"]   # if following 0, ratio is just follower count
    )
    return df.reset_index(drop=True)

wired_profiles = clean_profiles(wired_profiles_raw)
verge_profiles = clean_profiles(verge_profiles_raw)

# Filter profiles to those whose author actually appears in the cleaned post set
wired_profiles = wired_profiles[wired_profiles["did"].isin(wired_posts["author_did"])].reset_index(drop=True)
verge_profiles = verge_profiles[verge_profiles["did"].isin(verge_posts["author_did"])].reset_index(drop=True)

print(f"Wired profiles after alignment with cleaned posts     : {len(wired_profiles)}")
print(f"The Verge profiles after alignment with cleaned posts : {len(verge_profiles)}")

Wired profiles after alignment with cleaned posts     : 971
The Verge profiles after alignment with cleaned posts : 903


In [9]:
wired_posts.to_csv(PROC_DIR    / "wired_posts_clean.csv",    index=False)
verge_posts.to_csv(PROC_DIR    / "verge_posts_clean.csv",    index=False)
wired_profiles.to_csv(PROC_DIR / "wired_profiles_clean.csv", index=False)
verge_profiles.to_csv(PROC_DIR / "verge_profiles_clean.csv", index=False)

# Write a transparent audit JSON — quoting this in the methodology section
# directly addresses the rubric's "summary of process" criterion.
cleaning_summary = {
    "wired": {
        "filtering_audit": wired_audit,
        "n_posts_final": len(wired_posts),
        "n_unique_authors_final": int(wired_posts["author_did"].nunique()),
        "n_profiles_final": len(wired_profiles),
        "date_range": [str(wired_posts["created_at"].min()), str(wired_posts["created_at"].max())],
        "median_word_count": float(wired_posts["word_count_clean"].median()),
        "reply_share": float(wired_posts["is_reply"].mean()),
    },
    "verge": {
        "filtering_audit": verge_audit,
        "n_posts_final": len(verge_posts),
        "n_unique_authors_final": int(verge_posts["author_did"].nunique()),
        "n_profiles_final": len(verge_profiles),
        "date_range": [str(verge_posts["created_at"].min()), str(verge_posts["created_at"].max())],
        "median_word_count": float(verge_posts["word_count_clean"].median()),
        "reply_share": float(verge_posts["is_reply"].mean()),
    },
    "cleaning_steps_applied": [
        "URL/mention/hashtag extraction (preserved as separate columns)",
        "Light-clean for sentiment (URLs/mentions/hashtags stripped, casing kept)",
        "Aggressive clean for keyword/topic analysis (lowercase, no punct/digits/single letters, NLTK stopwords + custom stopwords, lemmatised via TextBlob)",
        "Brand-name stopword variant for topic modelling (text_clean_no_brand)",
        "Brand-relevance filter to remove idiom false positives ('on the verge of', 'fully wired', etc.)",
        "English-language filter (langs contains 'en')",
        "URI-level deduplication",
        "Same-author/same-text deduplication (repost spam)",
        "Minimum length filter (>=3 cleaned tokens)",
        "Author-level repeated-content spam filter (>5 identical posts)",
    ],
}

with open(PROC_DIR / "cleaning_summary.json", "w") as f:
    json.dump(cleaning_summary, f, indent=2)

print("Saved processed data + cleaning_summary.json")
print(json.dumps(cleaning_summary, indent=2))

Saved processed data + cleaning_summary.json
{
  "wired": {
    "filtering_audit": {
      "start": 1187,
      "after_language": 1187,
      "after_uri_dedup": 1187,
      "after_author_text_dedup": 1167,
      "after_min_length": 1136,
      "after_spam_filter": 1136,
      "after_timestamp_parse": 1090,
      "final": 1090
    },
    "n_posts_final": 1090,
    "n_unique_authors_final": 971,
    "n_profiles_final": 971,
    "date_range": [
      "2023-07-07 12:52:25.282000+00:00",
      "2026-04-29 16:37:42.505000+00:00"
    ],
    "median_word_count": 16.0,
    "reply_share": 0.4688073394495413
  },
  "verge": {
    "filtering_audit": {
      "start": 1250,
      "after_language": 1250,
      "after_uri_dedup": 1250,
      "after_author_text_dedup": 1246,
      "after_min_length": 1167,
      "after_spam_filter": 1167,
      "after_timestamp_parse": 1156,
      "final": 1156
    },
    "n_posts_final": 1156,
    "n_unique_authors_final": 903,
    "n_profiles_final": 903,
    "date_r

In [10]:
# These printouts go straight into your Data Description and Methodology sections.

def comparison_table(wired: pd.DataFrame, verge: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for label, df in [("Wired", wired), ("The Verge", verge)]:
        rows.append({
            "Brand": label,
            "Posts (final)": len(df),
            "Unique authors": df["author_did"].nunique(),
            "Posts/author": round(len(df) / max(df["author_did"].nunique(), 1), 2),
            "Replies (%)": round(df["is_reply"].mean() * 100, 1),
            "Mean likes": round(df["like_count"].mean(), 2),
            "Mean reposts": round(df["repost_count"].mean(), 2),
            "Median post length (clean tokens)": int(df["word_count_clean"].median()),
            "Posts with URLs (%)": round((df["n_urls"] > 0).mean() * 100, 1),
            "Posts with hashtags (%)": round((df["n_hashtags"] > 0).mean() * 100, 1),
        })
    return pd.DataFrame(rows).set_index("Brand")

summary_table = comparison_table(wired_posts, verge_posts)
print(summary_table.T)

# Save for the report
tables_dir = PROJECT_ROOT / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
summary_table.T.to_csv(tables_dir / "01_data_description_summary.csv")

Brand                                Wired  The Verge
Posts (final)                      1090.00    1156.00
Unique authors                      971.00     903.00
Posts/author                          1.12       1.28
Replies (%)                          46.90      37.40
Mean likes                           22.32       9.60
Mean reposts                          6.65       2.19
Median post length (clean tokens)    16.00      14.00
Posts with URLs (%)                  34.80      34.20
Posts with hashtags (%)               6.10      10.90
